<a href="https://colab.research.google.com/github/AAwaisYaseen/ai-avatar-communication-/blob/main/COMP6011_LLM_PoC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate bitsandbytes torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00


In [2]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from sklearn.metrics import accuracy_score, f1_score, classification_report

# loading hugging face token from colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')
print("Token loading check.")

Token loading check.


In [4]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

df = pd.read_excel('/content/drive/MyDrive/AAIRT-Task2/student_assignment_10_cases.xlsx',
                   sheet_name='Assignment_Cases')

print(f"Loaded {len(df)} cases successfully")
print(df[['Case ID', 'Risk Level']])

Mounted at /content/drive
Loaded 10 cases successfully
   Case ID Risk Level
0        1    attempt
1        2    attempt
2        3   behavior
3        4   behavior
4        5   ideation
5        6   ideation
6        7  indicator
7        8  indicator
8        9       safe
9       10       safe


In [7]:
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# import torch

# # 4-bit quantization config to fit in Colab T4 GPU
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16
# )

# llama_model_id = "meta-llama/Llama-3.1-8B-Instruct"

# print("Loading Llama 3.1 8B tokenizer...")
# llama_tokenizer = AutoTokenizer.from_pretrained(
#     llama_model_id,
#     token=HF_TOKEN
# )

# print("Loading Llama 3.1 8B model...")
# llama_model = AutoModelForCausalLM.from_pretrained(
#     llama_model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     token=HF_TOKEN
# )

# print("Llama 3.1 8B loaded successfully.")

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# using pre-quantized 4bit version which is less in
# size as the upper cell was bigger, at almost 16 GBs.
llama_model_id = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

print("loading Llama 3.1 8B tokenizer.")
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_id)

print("Loading Llama 3.1 8B model.")
llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Llama 3.1 8B loaded successfully.")

loading Llama 3.1 8B tokenizer.


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Llama 3.1 8B loaded successfully.


In [8]:
# def classify_risk(model, tokenizer, conversation):

#     prompt = f"""You are helping assess suicide risk in a clinical setting.
# I need you to read the conversation below between a person and an AI support agent,
# and tell me what level of suicide risk the person is showing.

# Here are the five risk levels you can choose from:
# - attempt: the person has made or is actively planning a suicide attempt
# - behavior: the person is doing things that suggest they may attempt suicide soon, like giving away belongings or saying goodbye
# - ideation: the person is having thoughts of suicide but has no clear plan
# - indicator: the person is showing signs of serious distress like hopelessness or feeling like a burden, but no direct suicidal thoughts
# - safe: the person is not showing any signs of suicide risk

# Read the conversation carefully and reply with only one word from the list above.

# Conversation:
# {conversation}

# Risk level:"""

#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=10,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id
#         )

#     full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     response = full_response[len(prompt):].strip().lower()
#     response = response.split()[0] if response else "unknown"

#     valid_labels = ["attempt", "behavior", "ideation", "indicator", "safe"]
#     if response not in valid_labels:
#         response = "unknown"

#     return response

# print("ready...")

ready...


In [18]:
def classify_risk(model, tokenizer, conversation):

    prompt = f"""Read the conversation and classify the suicide risk level.

Definitions:
- attempt: person has made or is actively planning a suicide attempt
- behavior: person is doing things suggesting they may attempt suicide soon
- ideation: person is having thoughts of suicide but no clear plan
- indicator: person is showing serious distress like hopelessness but no direct suicidal thoughts
- safe: no signs of suicide risk

Conversation:
{conversation}

Reply with one word only. One word. No sentences. No explanation. Just the label:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

    valid_labels = ["attempt", "behavior", "ideation", "indicator", "safe"]
    for label in valid_labels:
        if label in response:
            return label

    return "unknown"

print("Function defined.")

Function defined.


In [19]:
# running all of the 10 cases with Llama 3.1 8B
print("Running 10 cases...\n")

llama_predictions = []

for index, row in df.iterrows():
    case_id = row['Case ID']
    conversation = row['Paraphrased Dialogue']
    ground_truth = row['Risk Level']

    predicted = classify_risk(llama_model, llama_tokenizer, conversation)
    llama_predictions.append(predicted)

    print(f"Case {case_id} | Ground Truth: {ground_truth} | Predicted: {predicted}")

print("\nDone. All 10 cases.")

Running 10 cases...

Case 1 | Ground Truth: attempt | Predicted: ideation
Case 2 | Ground Truth: attempt | Predicted: attempt
Case 3 | Ground Truth: behavior | Predicted: attempt
Case 4 | Ground Truth: behavior | Predicted: indicator
Case 5 | Ground Truth: ideation | Predicted: indicator
Case 6 | Ground Truth: ideation | Predicted: attempt
Case 7 | Ground Truth: indicator | Predicted: ideation
Case 8 | Ground Truth: indicator | Predicted: attempt
Case 9 | Ground Truth: safe | Predicted: behavior
Case 10 | Ground Truth: safe | Predicted: behavior

Done. All 10 cases.


In [7]:
import nbformat

notebook_path = "/content/drive/MyDrive/Colab Notebooks/COMP6011_LLM_PoC.ipynb"

with open(notebook_path, 'r') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

with open(notebook_path, 'w') as f:
    nbformat.write(nb, f)

print("fix check")

fix check
